# Full Factorial: Illustration of the scaling law of recurrent networks.

The aim of this notebook is to illustrate the effect on training process of hyperparameters changes.

## The Hyperparameters

The hyperparameters that we will consider are:

- Updates: the number of optimizer updates the model will take. One epoch performs
  `ceil(Split ratio * Number of sequences / Batch size)` updates - the split and the
  short final batch both matter - so *Epochs = ceil(Updates / updates per epoch)*.
- Sequence length: axes 1 of the input data, the length of the input sequences. (characters length, must be even: a term is 2 characters)
- Number of sequences: axes 0 of the input data, i.e. the number of sequences in the dataset. Half are generated from the grammar, half uniformly from the alphabet, and every label is obtained by parsing.


### Static Hyperparameters

The following hyperparameters will remain constant throughout the notebook:
- Batch size
- Learning rate (cosine schedule, restarted at each checkpoint - see below)
- Token dimension
- Split ratio (train/test)
- Seed (each cell is reseeded, so the dataset is a controlled variable rather than noise)

## Experiment

The analysis is performed through the full factorial technique over the three
factors *updates*, *number of sequences* and *sequence length*.

The grid is only swept over **number of sequences x sequence length** (9 runs). The
*updates* factor is not swept: each run is trained to the largest budget and a
checkpoint is harvested at every smaller budget along the way, giving the full
3 x 3 x 3 = 27 design points for the cost of the largest budget alone.

That substitution is only sound if a harvested checkpoint is interchangeable with a
run of that budget. A single cosine spanning the whole run would break it - the
512-update checkpoint would still be near `MAX_LR` while the 2048-update one had
annealed to `MIN_LR`, so the updates factor would be confounded with the learning
rate. The schedule therefore **restarts the cosine at each checkpoint**, annealing
`MAX_LR -> MIN_LR` within every segment, so each checkpoint is a fully annealed
model of its own budget.

### Checkpoint recovery

Artifacts are named by a deterministic experiment id, so an interrupted notebook can
be re-run: completed cells are skipped, and a partially finished cell resumes from
its last checkpoint instead of retraining. A checkpoint only counts as complete when
all three of its artifacts (backup, training metrics, validation metrics) are present.

## The Training Problem

Given a language grammar, the LSTM will be able to classify sequences of characters as valid or invalid according to the grammar rules.

BNF Definition:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, OK \\[2pt]
\end{array}
$$

In [1]:
"""Static Hyperparameters Configuration"""

from examples.recurrent.scaling_law.helpers import Experiment

MAX_LR, MIN_LR = 1e-2, 1e-4

## Each experiment reseeds from Experiment.seed, so re-running - or resuming - a cell
## rebuilds exactly the same dataset. Without it the difference between two factor
## levels is confounded with the difference between two random data draws.
experiment = Experiment(batch_size=16, split_ratio=0.9, seed=777)

ImportError: cannot import name 'Artifact' from 'thorcino.artifact' (/home/cecinuga/Scrivania/Github/thorcino/thorcino/artifact/__init__.py)

In [ ]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import BinaryCrossEntropyLoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineRestartSchedule
from thorcino.training.trainer import Trainer

def make_trainer(checkpoint_epochs: list[int]) -> Trainer:
    """Build a trainer whose schedule restarts at every checkpoint.

    The updates factor is read off checkpoints of a single run rather than from one
    run per budget. That only measures what it claims to if the learning rate at a
    checkpoint does not depend on how long the run continues afterwards: with one
    cosine spanning the whole run, the 512-update checkpoint would still sit near
    MAX_LR while the 2048-update one had annealed to MIN_LR, and the comparison
    would confound the update budget with the learning rate.

    Restarting the cosine at each checkpoint anneals MAX_LR -> MIN_LR within every
    segment, so each harvested checkpoint is a fully annealed model of its budget.
    """
    model = Sequential(
        LSTM(in_feature=3, hidden_units=3, out_type='n_to_1'),
        Linear(in_feature=3, out_feature=1),
        Sigmoid(),
    )
    loss = BinaryCrossEntropyLoss()
    optimizer = SGD(model.parameters, MAX_LR)
    scheduler = CosineRestartSchedule(MAX_LR, MIN_LR, [e + 1 for e in checkpoint_epochs])

    return Trainer(model, loss, optimizer, scheduler)

## Validation Set

A validation set is used to evaluate the model performance in order to choose the best hyperparameters configuration.
The validation set is shared across all runs.

Considering that the problem is length independent, several sub validation set are build, each set has a different sequence length, this last, increase by a factor of 2, starting from the minimum sequence length used in the experiments.

In [ ]:
"""Create the Validation Set"""

VAL_SEQUENCE_LENGTHS = [8, 16, 32, 64, 96]

val_dataset = experiment.validation_set(500, VAL_SEQUENCE_LENGTHS)

validation set: SEQUENCE_LENGTH=8 X=(500, 8, 3) Y=(500, 1) positives=0.500
validation set: SEQUENCE_LENGTH=16 X=(500, 16, 3) Y=(500, 1) positives=0.500
validation set: SEQUENCE_LENGTH=32 X=(500, 32, 3) Y=(500, 1) positives=0.500
validation set: SEQUENCE_LENGTH=64 X=(500, 64, 3) Y=(500, 1) positives=0.500
validation set: SEQUENCE_LENGTH=96 X=(500, 96, 3) Y=(500, 1) positives=0.500


## The Training Process

`Experiment.run_full_factorial` trains one run per grid cell, up to the largest update
budget. At every budget in `hyperparams['updates']` three artifacts are written: the
full checkpoint (model, optimizer and scheduler state), the training metrics, and the
scores against the shared validation set.

The validation sweep is evaluated with `record=False` so it does not append to the same
history the training curve is read from; the periodic test evaluation every `eval_step`
epochs is the only thing that does.

In [ ]:
"""Running the Full Factorial"""

hyperparams = {
    'updates': [512, 1024, 2048],
    'number_of_sequence': [128, 256, 512],
    'sequence_length': [8, 16, 32]
}

completed_checkpoints = experiment.run_full_factorial(hyperparams, make_trainer, val_dataset)

recovered 27 completed checkpoint(s): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=256 UPDATES=2048, NUMBER_OF_SEQUENCE=128, SEQUENCE_LENGTH=8
updates per epoch: 8, checkpoint epochs: [63, 127, 255]
all 3 checkpoints already saved, skipping
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=256 UPDATES=2048, NUMBER_OF_SEQUENCE=128, SEQUENCE_LENGTH=16
updates per epoch: 8, checkpoint epochs: [63, 127, 255]
all 3 checkpoints already saved, skipping
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=256 UPDATES=2048, NUMBER_OF_SEQUENCE=128, SEQUENCE_LENGTH=32
updates per epoch: 8, checkpoint epochs: [63, 127, 255]
all 3 checkpoints already saved, skipping

In [ ]:
from examples.recurrent.scaling_law.helpers import load_experiment_artifact, log_scale


log_updates = log_scale(hyperparams['updates'])
log_n_seq = log_scale(hyperparams['number_of_sequence'])
log_s_len = log_scale(hyperparams['sequence_length'])

experiment_metrics = load_experiment_artifact(f'{experiment.training_folder}/E0__64_512_128_8__11s.pkl')
print(experiment_metrics)

ExperimentArtifact(data=Artifact(data={'epoch': 64, 'step': 960, 'history': {'train_loss': [0.6939240097999573, 0.6937812288602193, 0.6932103117307027, 0.6926130374272664, 0.6930056969324748, 0.6917847553888957, 0.6917095104853312, 0.6922126889228821, 0.6914138634999593, 0.6910813132921855, 0.690557070573171, 0.6903200705846151, 0.6890345970789592, 0.6889805555343628, 0.6885244607925415, 0.687705667813619, 0.6875007629394532, 0.6872871081034343, 0.6871476928393047, 0.6865299701690674, 0.6862205664316813, 0.6867072224617005, 0.6869306643803914, 0.6860363642374675, 0.6863837679227193, 0.6864109595616659, 0.6849636912345887, 0.6849490563074748, 0.6862694422403971, 0.6858222246170044, 0.6853242158889771, 0.6851815740267436, 0.6851900458335877, 0.6845059911410014, 0.6853691299756368, 0.6845131754875183, 0.6840879480044048, 0.6839078744252522, 0.6838571429252625, 0.6849064270655314, 0.6842260599136353, 0.6843992034594218, 0.6840608239173889, 0.6839425166447958, 0.6836848696072896, 0.68409810